# Clustering Analysis on doctors_aggregated.csv
## 1. Imports y Configuración


In [ ]:
%matplotlib widget
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from warnings import filterwarnings
filterwarnings('ignore')

# Use sklearn-extra for K-Medoids if available
try:
    from sklearn_extra.cluster import KMedoids
except ImportError:
    print("Por favor instala scikit-learn-extra para K-Medoids: pip install scikit-learn-extra")
    KMedoids = None


## 2. Carga, Preprocesamiento y Muestreo (Same Samples)
Debido al costo O(N^2) de K-Medoids y Silhouette, fijaremos un random_state y seleccionaremos una muestra representativa (ej. 5,000 registros).


In [ ]:
# Cargar datos
df = pd.read_csv('doctors_aggregated.csv')

# Filtrar columnas texto y de IDs
cols_to_drop = ['NUEVO_ID', 'ATSEG_first', 'WEEK_ID_first', 'WEEK_ID_last']
features_df = df.drop(columns=[c for c in cols_to_drop if c in df.columns]).fillna(0)
# Eliminar posibles textos residuales
features_df = features_df.select_dtypes(include=[np.number])

# Balanceo de las características usando RobustScaler para limitar efecto de outliers
scaler = RobustScaler()
X_scaled = scaler.fit_transform(features_df)

# Guardar la transformación en un DF
df_scaled = pd.DataFrame(X_scaled, columns=features_df.columns, index=features_df.index)

# Sampleo balanceado (Misma muestra para todos los algoritmos)
N_SAMPLES = min(5000, len(df_scaled))
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
sample_indices = np.random.choice(df_scaled.index, size=N_SAMPLES, replace=False)

# Datos que usaran los algoritmos
X_sample = df_scaled.loc[sample_indices].values
original_sample = df.loc[sample_indices].copy()

print(f"Dataset original: {df.shape}")
print(f"Sample utilizado para Clustering: {X_sample.shape}")


## 3. Selección de $k$ y Comparación con $k=3$


In [ ]:
# Evaluaremos K-Means para distintos k
k_values = range(2, 9)
inertias = []
sil_scores = []
db_scores = []
ch_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init='auto')
    labels = km.fit_predict(X_sample)
    
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_sample, labels))
    db_scores.append(davies_bouldin_score(X_sample, labels))
    ch_scores.append(calinski_harabasz_score(X_sample, labels))

# Encontrar k optima según silhouette max imo
optimal_k = k_values[np.argmax(sil_scores)]

plt.figure(figsize=(15, 4))
plt.subplot(1,3,1)
plt.plot(k_values, inertias, marker='o')
plt.title("Elbow Method (Inertia)")
plt.xlabel("k")

plt.subplot(1,3,2)
plt.plot(k_values, sil_scores, marker='o', color='green')
plt.title("Silhouette Score (Max is better)")
plt.axvline(x=3, color='red', linestyle='--', label='k=3')
plt.axvline(x=optimal_k, color='purple', linestyle='--', label=f'k={optimal_k} (Optimal)')
plt.legend()
plt.xlabel("k")

plt.subplot(1,3,3)
plt.plot(k_values, db_scores, marker='o', color='orange')
plt.title("Davies-Bouldin (Min is better)")
plt.axvline(x=3, color='red', linestyle='--')
plt.axvline(x=optimal_k, color='purple', linestyle='--')
plt.xlabel("k")
plt.tight_layout()
plt.show()

# Comparación k=3 vs Optimal k
print(f"Mejor K analítico según Silhouette: {optimal_k}")
comp_df = pd.DataFrame({
    'Metric': ['Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz'],
    'k=3': [sil_scores[1], db_scores[1], ch_scores[1]],
    f'k={optimal_k} (Optimal)': [sil_scores[optimal_k-2], db_scores[optimal_k-2], ch_scores[optimal_k-2]]
})
display(comp_df)


## 4. Ejecución de Clustering con $k=3$ y PCA


In [ ]:
K = 3
results = {}

# K-Means
kmeans = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init='auto')
results['K-Means'] = kmeans.fit_predict(X_sample)

# K-Medoids
if KMedoids is not None:
    kmedoids = KMedoids(n_clusters=K, random_state=RANDOM_STATE, method='pam')
    results['K-Medoids'] = kmedoids.fit_predict(X_sample)

# Hierarchical (Ward)
hier = AgglomerativeClustering(n_clusters=K, linkage='ward')
results['Hierarchical'] = hier.fit_predict(X_sample)

# GMM
gmm = GaussianMixture(n_components=K, random_state=RANDOM_STATE)
results['GMM'] = gmm.fit_predict(X_sample)

# DBSCAN (no requiere K)
dbscan = DBSCAN(eps=25, min_samples=10) # parametros arbitrarios aprox para ilustrar
results['DBSCAN'] = dbscan.fit_predict(X_sample)

# Agregamos las predicciones originales al df
for model_name, lab in results.items():
    original_sample[f'Cluster_{model_name}'] = lab
    
# PCA para visualización (2 componentes)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_sample)
original_sample['PCA1'] = X_pca[:, 0]
original_sample['PCA2'] = X_pca[:, 1]
print(f"Varianza explicada por las 2 primeras componentes PCA: {pca.explained_variance_ratio_.sum()*100:.2f}%")


## 5. Visualización: Clústeres, Centroides y Medoides en Espacio PCA


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

models_to_plot = ['K-Means', 'K-Medoids', 'Hierarchical', 'GMM']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for ax, model in zip(axes, models_to_plot):
    if model not in results:
        continue
    
    labels = results[model]
    
    # Scatter points
    for k_idx in range(K):
        mask = (labels == k_idx)
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], s=10, alpha=0.5, color=colors[k_idx], label=f'Cluster {k_idx}')
        
    # Destacar Centroides / Medoides dependiendo del modelo
    if model == 'K-Means':
        # kmeans.cluster_centers_ exist in high dimension, project to PCA
        centers_pca = pca.transform(kmeans.cluster_centers_)
        ax.scatter(centers_pca[:, 0], centers_pca[:, 1], c='black', marker='X', s=200, edgecolors='white', label='Centroids')
        
    elif model == 'K-Medoids':
        # kmedoids centroids are actual points index, we can just grab their PCA coords
        medoid_indices = kmedoids.medoid_indices_
        ax.scatter(X_pca[medoid_indices, 0], X_pca[medoid_indices, 1], c='black', marker='D', s=150, edgecolors='white', label='Medoids')
        
    elif model == 'GMM':
        # Means of the gaussians
        centers_pca = pca.transform(gmm.means_)
        ax.scatter(centers_pca[:, 0], centers_pca[:, 1], c='black', marker='X', s=200, edgecolors='white', label='GMM Means (Centroids)')
        
    ax.set_title(f'{model} (k=3)')
    ax.set_xlabel('PCA Component 1')
    ax.set_ylabel('PCA Component 2')
    ax.legend(loc='best')

plt.tight_layout()
plt.show()


## 6. Comparación y Tabuleo de $k=3$ para todos los Algoritmos


In [ ]:
metrics_k3 = []
for model_name, lab in results.items():
    if len(set(lab)) > 1: # Si no es un solo cluster (ej DBSCAN noise)
        try:
            sil = silhouette_score(X_sample, lab)
            ch = calinski_harabasz_score(X_sample, lab)
            db = davies_bouldin_score(X_sample, lab)
        except:
            sil, ch, db = np.nan, np.nan, np.nan
        metrics_k3.append({
            'Algorithm': model_name,
            'Silhouette': sil,
            'Calinski-Harabasz': ch,
            'Davies-Bouldin': db
        })

df_metrics_k3 = pd.DataFrame(metrics_k3)
display(df_metrics_k3.sort_values(by='Silhouette', ascending=False))
